# Section 2 — NTM: Read and Write from Scratch


But attention can only **read** from the tokens already in the context window. What if we could also **write** — update this memory during inference, outside the context window entirely?

We build the core NTM memory module piece by piece: matrix, addressing, read, write.

## 2b — Content-Based Addressing

To read or write, we first ask: which slot is most similar to my query? The answer is a soft probability distribution over all slots — not a hard index.

In [ ]:
def content_address(query: torch.Tensor, memory: torch.Tensor) -> torch.Tensor:
    # cosine similarity between query and each memory slot
    sim: torch.Tensor = F.cosine_similarity(query.unsqueeze(0), memory, dim=1)
    # softmax converts similarities to a probability distribution over slots
    weights: torch.Tensor = F.softmax(sim, dim=0)
    return weights


# demo: random query against empty memory → uniform weights
torch.manual_seed(7)
test_query: torch.Tensor = torch.randn(VEC_DIM)
addr_weights: torch.Tensor = content_address(test_query, memory)

# visualize address weights as a bar chart
fig, ax = plt.subplots(figsize=(7, 3))
colors: np.ndarray = plt.cm.Blues(
    (addr_weights.detach().numpy() - addr_weights.min().item()) /
    (addr_weights.max().item() - addr_weights.min().item() + 1e-9)
)
ax.bar(range(N_SLOTS), addr_weights.detach().numpy(), color=colors, edgecolor="#333")
ax.set_xlabel("Memory slot")
ax.set_ylabel("Address weight")
ax.set_title("Content-based address weights on empty memory — near-uniform distribution")
ax.set_xticks(range(N_SLOTS))
plt.tight_layout()
plt.show()

print(f"Weights sum to: {addr_weights.sum().item():.4f}  (should be 1.0)")
print(f"Min: {addr_weights.min().item():.4f}  Max: {addr_weights.max().item():.4f}")
print("Empty memory → all slots look equally similar → uniform distribution")

## 2c — Memory Read

Reading from memory: take the weighted sum of all slots, where the weights come from addressing.

In [ ]:
def memory_read(weights: torch.Tensor, memory: torch.Tensor) -> torch.Tensor:
    # weighted combination of all memory slots
    retrieved: torch.Tensor = weights @ memory   # shape: (VEC_DIM,)
    return retrieved


# quick sanity check on empty memory
retrieved_vec: torch.Tensor = memory_read(addr_weights, memory)
print(f"Read from empty memory → norm of retrieved vector: {retrieved_vec.norm().item():.4f}")
print("(Should be ~0: nothing stored yet)")

## 2d — Memory Write

Writing to memory uses two vectors: an **erase** vector (what to forget) and an **add** vector (what to write). Both are applied proportionally to the address weights.

In [ ]:
def memory_write(
    memory: torch.Tensor,
    weights: torch.Tensor,
    erase: torch.Tensor,
    add: torch.Tensor
) -> torch.Tensor:
    # erase: selectively reduce content in addressed slots
    # weights.unsqueeze(1) broadcasts over vector dimension
    memory_erased: torch.Tensor = memory * (1 - weights.unsqueeze(1) * erase.unsqueeze(0))
    # add: write new content into addressed slots
    memory_written: torch.Tensor = memory_erased + weights.unsqueeze(1) * add.unsqueeze(0)
    return memory_written

## 2e — Demo: Write 5 Facts, Then Query

Now the full demo. We write 5 synthetic 'facts' into memory — each fact is a (key, value) pair encoded as latent vectors. Then we query with exact keys, noisy keys, and an orthogonal key.

In [ ]:
torch.manual_seed(0)

N_FACTS: int = 5

# --- generate 5 random unit-normalized key vectors and value vectors ---
raw_keys: torch.Tensor = torch.randn(N_FACTS, VEC_DIM)
fact_keys: torch.Tensor = F.normalize(raw_keys, dim=1)          # unit vectors
fact_values: torch.Tensor = torch.randn(N_FACTS, VEC_DIM) * 0.8

# erase vector: all ones = full overwrite (writing into fresh slots)
erase_vec: torch.Tensor = torch.ones(VEC_DIM)

# --- write each (key → value) pair into memory using the key as the address ---
mem: torch.Tensor = torch.zeros(N_SLOTS, VEC_DIM)
for i in range(N_FACTS):
    w: torch.Tensor = content_address(fact_keys[i], mem)
    mem = memory_write(mem, w, erase_vec, fact_values[i])

# --- visualize memory matrix after all writes ---
fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(mem.detach().numpy(), cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
ax.set_xlabel("Vector dimension")
ax.set_ylabel("Memory slot")
ax.set_title("Memory matrix M after writing 5 facts")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# --- define four query variants ---
query_exact: torch.Tensor = fact_keys[0]
query_slight: torch.Tensor = F.normalize(fact_keys[0] + 0.3 * torch.randn(VEC_DIM), dim=0)
query_heavy: torch.Tensor = F.normalize(fact_keys[0] + 1.5 * torch.randn(VEC_DIM), dim=0)

# orthogonal key: use a random vector orthogonalized to key[0]
rand_vec: torch.Tensor = torch.randn(VEC_DIM)
rand_vec = rand_vec - (rand_vec @ fact_keys[0]) * fact_keys[0]
query_orthogonal: torch.Tensor = F.normalize(rand_vec, dim=0)

queries: list[tuple[str, torch.Tensor]] = [
    ("Exact key", query_exact),
    ("Slight noise (\u03c3=0.3)", query_slight),
    ("Heavy noise (\u03c3=1.5)", query_heavy),
    ("Orthogonal key", query_orthogonal),
]

# --- plot address weight distributions for all four queries ---
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
axes_flat: list = axes.flatten().tolist()

for ax, (title, q) in zip(axes_flat, queries):
    w: torch.Tensor = content_address(q, mem)
    w_np: np.ndarray = w.detach().numpy()
    # normalize for colormap so highest bar is darkest blue
    normed: np.ndarray = (w_np - w_np.min()) / (w_np.max() - w_np.min() + 1e-9)
    bar_colors: np.ndarray = plt.cm.Blues(0.3 + 0.7 * normed)
    ax.bar(range(N_SLOTS), w_np, color=bar_colors, edgecolor="#333")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Memory slot")
    ax.set_ylabel("Address weight")
    ax.set_xticks(range(N_SLOTS))
    ax.set_ylim(0, None)

fig.suptitle("Address weights for four query variants (target: slot 0)", fontsize=13)
plt.tight_layout()
plt.show()

# Section 3 — The Dictionary Contrast

Let's make the difference between NTM memory and a Python dictionary brutally clear.

In [ ]:
# --- Python dictionary: exact lookup only ---
facts: dict[str, str] = {
    "capital of France": "Paris",
    "boiling point of water": "100\u00b0C",
}

# exact key works
print(f"Exact:  {facts['capital of France']}")

# noisy key fails
noisy_key: str = "capitel of Frence"
try:
    print(facts[noisy_key])
except KeyError as e:
    print(f"KeyError: {e}")

In [ ]:
# --- NTM lookup: use a noisy version of fact_keys[0] as the query ---
# (pretend fact_keys[0] encodes 'capital of France')
torch.manual_seed(99)
noisy_query: torch.Tensor = F.normalize(fact_keys[0] + 0.4 * torch.randn(VEC_DIM), dim=0)

# address and retrieve from the NTM memory populated in Section 2e
noisy_weights: torch.Tensor = content_address(noisy_query, mem)
retrieved_noisy: torch.Tensor = memory_read(noisy_weights, mem)

# measure correctness: cosine similarity to the stored value
correct_value: torch.Tensor = fact_values[0]
similarity: float = float(
    F.cosine_similarity(retrieved_noisy.unsqueeze(0),
                        correct_value.unsqueeze(0)).item()
)

print(f"NTM retrieved a vector of norm {retrieved_noisy.norm().item():.3f}")
print(f"Cosine similarity to correct value vector: {similarity:.3f}")
print("(1.0 = perfect recall, 0.0 = unrelated)")

**Is that remembering? You decide.**

<details>
<summary>what 'reasonable retrieval' actually means</summary>

The NTM does not return the string "Paris". It returns a vector — the same kind of high-dimensional floating-point object that flows through every layer of a neural network. We cannot read this vector the way we read text. What we *can* do is measure its geometric relationship to the stored value vector using cosine similarity.

A similarity of 1.0 means the retrieved vector points in exactly the same direction as the stored value — perfect recall. A similarity of 0.0 means the two vectors are orthogonal — the retrieved vector shares nothing with the stored one. In practice, a noisy query returns a retrieval that sits somewhere between these extremes: recognizably related to the correct answer, degraded by the noise in the cue.

This is why the latent representation matters. If memory stored tokens, we would need to decode the retrieved vector back into words to evaluate it — a lossy and ambiguous step. Because both the stored value and the retrieved vector live in the same continuous space, we can evaluate recall quantitatively, without ever leaving the representational substrate of the network.

</details>

<details>
<summary>the situated representation problem — what it means for memory design</summary>

This is not a technical curiosity — it is a design choice that every memory architecture must answer, explicitly or implicitly.

NTM leaves the choice to the controller: whichever vector the controller produces as its query or add signal is what gets written. The controller learns, from training, which representation is most useful to store. In practice this tends to be something like the hidden state after processing a token — already contextualized by self-attention, but before any output projection.

Memorizing Transformers make a more constrained choice: what gets cached are the K and V projections of each token at a specific layer. These are layer-specific learned linear transformations of the attention input. The geometry of the cached space is the geometry of that layer’s key-value space — which is what makes retrieval by dot-product coherent.

Titans writes into a small MLP whose weights are the memory, so the vector question does not apply in the same way. What gets encoded is implicitly whatever the MLP’s weights, after a gradient step, represent. Less interpretable, but potentially more expressive.

The practical upshot: when you hear “the model stored X in memory,” always ask — at which layer, in which projection, at which position in the sequence? The answer changes what “X” actually is.

</details>


<details>
<summary>the erase-then-add mechanism</summary>

Why not simply overwrite the slot? Pure overwrite (assigning `memory[slot] = add`) would destroy all previous content in that slot, even dimensions that were storing unrelated useful information. Memory slots are vectors of many dimensions, and the controller may only want to update a subset of those dimensions.

The erase vector solves this. Each element of the erase vector is a number in [0, 1] (produced by a sigmoid), controlling how much of each dimension to clear before writing. An erase element of 1 fully clears that dimension in the addressed slots. An element of 0 leaves it intact. The add vector then writes new content only into the cleared dimensions.

This gives the controller fine-grained control: it can update one dimension of a slot while preserving others. It is the difference between `dict[key] = new_value` and `dict[key].update({specific_field: new_value})`. The latter is strictly more expressive — and that expressiveness is what makes NTM capable of complex algorithmic tasks like sorting and copying.

</details>

<details>
<summary>graceful degradation — why this matters</summary>

A hash table either returns the stored value exactly, or raises a KeyError. There is no in-between. The NTM's content-based addressing has no such cliff: as the query drifts away from the stored key, the address distribution broadens gradually. The system still returns *something* — a blend weighted by similarity to all stored patterns.

This is not a failure mode to be engineered away. It is the defining feature of associative memory. Human memory works the same way: a noisy or partial cue ('what was that song that goes...') still retrieves something plausible, often correct, sometimes a near-miss. The reconstruction is approximate but guided.

For agents operating in noisy environments — parsing imperfect tool outputs, handling ambiguous user instructions, reasoning over partially-observed state — graceful degradation under query noise is strictly more useful than exact-match-or-fail. The NTM's memory model is, in this specific sense, more robust than a dictionary.

</details>

<details>
<summary>why the same vocabulary matters</summary>

In NTM, memory access is a separate mechanism bolted onto the model — a controller with dedicated read and write heads that use a different addressing scheme than the main attention layers. It works, but there is a seam: the model has two representational worlds and must translate between them.

Memorizing Transformers (Wu et al., 2022) remove the seam. The external memory bank stores (K, V) pairs in exactly the same format as the local context. The retrieval operation is exactly the same scaled dot-product attention the model uses for everything else. There is no mode switch, no separate read head, no translation step.

This means that when you extend a standard transformer with a Memorizing Transformer memory bank, you are not adding a new capability — you are extending an existing one. The model's internal language (the geometry of its key and query spaces) does not change. Past context and current context are both keys and values; the model cannot tell — and does not need to know — which side of the separator line a particular key came from.

</details>

The boundary between local context and retrieved memory is architectural, not representational. The model does not know — or care — which side of the line a key came from.